# 1020 33일차

## 1. Tool Calling

모델이 외부 기능(도구)을 쓰게 하는 방법 (`lg05_ToolNode`)

학습 데이터 이후의 정보(오늘 주가 등)는 모델 답에 들어갈 수 없어서, 검색 같은 도구로 가져와 넘겨야 함

1. 도구 : 이름·설명·인자를 가진 기능. 웹 검색 도구 등
2. `bind_tools(tools)` : 모델에 도구 목록(이름·설명·인자)을 알려줌
3. `tool_calls` : 모델이 만든 도구 호출 요청. 도구 이름과 인자가 들어 있음

- 모델은 도구를 직접 실행하지 않고 `tool_calls`만 만듦
- 도구가 여러 개면 모델이 이름과 설명을 보고 고름 (`lg06_02_ddg`)

## 2. 검색 도구

1. `TavilySearch` : `.env`에 `TAVILY_API_KEY`가 있어야 함. 결과는 질문과 `results`(`url`·`title`·`content`·`score`) dict
2. `DuckDuckGoSearchRun` : API 키 없이 씀(`ddgs` 패키지). 결과는 검색 요약문을 이어 붙인 문자열

- `max_results` : 가져올 검색 결과 수
- Tavily는 페이지 본문까지 넘겨서 모델에 들어가는 토큰이 DuckDuckGo보다 많음

#### cf) DuckDuckGo의 max_results

- `DuckDuckGoSearchRun(max_results=5)`처럼 넣으면 에러 없이 무시되고 기본값 5개로 고정
- 개수는 `api_wrapper=DuckDuckGoSearchAPIWrapper(max_results=5)`로 넘겨야 적용됨

## 3. ToolNode

`tool_calls`대로 도구를 실행하는 노드 (`lg05_ToolNode`)

모델이 도구를 쓰겠다고 할 때만 도구를 실행하고, 결과를 다시 모델에 넘기려고 조건부 엣지와 같이 씀

1. `ToolNode(tools)` : 도구를 실행해 결과를 `ToolMessage`로 `messages`에 붙임
2. `add_conditional_edges(노드, 함수)` : 함수가 돌려준 이름의 노드로 이동
3. `tools_condition` : 마지막 메시지에 `tool_calls`가 있으면 `"tools"`, 없으면 `END`를 돌려줌
4. `add_edge("tools", "chatbot")` : 도구 결과를 들고 모델로 돌아감

- 흐름은 `START → chatbot → tools → chatbot → … → END`, `tool_calls`가 없을 때까지 chatbot과 tools를 반복
- `tools_condition`이 `"tools"`를 돌려주므로 노드 이름을 `tools`로 등록
- `END`로 가는 길은 `tools_condition`에 들어 있어서 `add_edge("chatbot", END)`를 따로 쓰지 않음

## 4. 그래프 그림

컴파일한 그래프의 노드와 엣지를 그림으로 그리는 방법 (`lg07_ToolNode_graph`)

노드와 엣지가 늘어나면 코드만으로는 흐름을 확인하기 어려워서, 그림으로 연결을 확인함

1. `get_graph()` : 컴파일한 그래프의 구조
2. `draw_mermaid_png()` : 구조를 그린 PNG 바이트. 기본은 mermaid.ink 웹 API로 그려서 인터넷이 필요함
3. `draw_mermaid()` : mermaid 문법 텍스트. 인터넷 없이 확인

- 실선은 `add_edge`, 점선은 조건부 엣지
- PNG 바이트는 `open(경로, "wb")`로 저장. `with` 블록이 끝나면 파일이 자동으로 닫힘